# eph_01 — Monovariate RT encoding

Does reaction time predict spike count?

**Pipeline:**
1. Data loading via `data_loading.py`
2. Trial × unit table via `ephys_utils.build_all_counts_df`
3. `AnalysisSpec` + `fit_encoding` — OLS and Spearman, filtering external
4. `PerUnitStatsRegistry` — standard output for downstream spatial / comparison notebooks
5. Diagnostics: T distributions, RT window sweep, population tuning curve, early vs late


## 1. Setup

In [ ]:
%matplotlib inline

import contextlib, io, itertools, pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
mpl.rcParams["svg.fonttype"] = "none"

from statsmodels.nonparametric.smoothers_lowess import lowess as sm_lowess

In [ ]:
# ── Environment detection ─────────────────────────────────────────────────
from pathlib import Path

if Path("/root/capsule").exists():
    ENV       = "codeocean"
    SCRATCH   = Path("/root/capsule/scratch")
    DATA_ROOT = Path("/root/capsule/data")
    FOR_LOCAL = SCRATCH / "for_local"
    MESH_PATH = DATA_ROOT / "LC_percentile_meshes/new_core_mesh.obj"
else:
    ENV       = "local"
    FOR_LOCAL = Path("/Users/mib/Documents/Code/kinematics_analysis/data/for_local")
    DATA_ROOT = FOR_LOCAL
    SCRATCH   = FOR_LOCAL.parent
    MESH_PATH = FOR_LOCAL / "new_core_mesh.obj"

FIG_DIR  = SCRATCH / "figures" / "eph_01_rt_encoding"
SAVE_FIG = False

print(f"ENV       : {ENV}")
print(f"FOR_LOCAL : {FOR_LOCAL}")

## 2. Data loading

In [ ]:
from data_loading import (
    load_session_quality_filter,
    filter_ephys_units,
    load_units_with_spike_times,
)

if ENV == "codeocean":
    base_dirs = [SCRATCH / "session_analysis_mlk"]
    filtered_session_paths = load_session_quality_filter(base_dirs)

    with open(SCRATCH / "combined_unit_tbl.pkl", "rb") as f:
        combined_ephys_data = pickle.load(f)
    filtered_ephys = filter_ephys_units(combined_ephys_data, filtered_session_paths)

    ROOT_SCRATCH = str(DATA_ROOT / "LC-NE_scratch_data_1")
    units_with_spikes = load_units_with_spike_times(filtered_ephys, ROOT_SCRATCH)
else:
    # Local dev: load saved intermediates
    filtered_ephys    = pd.read_pickle(FOR_LOCAL / "filtered_ephys.pkl")
    units_with_spikes = None   # not needed locally; all_counts_df loaded below
    base_dirs         = [FOR_LOCAL]
    print(f"Local dev: filtered_ephys {filtered_ephys.shape}")

## 3. Build trial × unit table (all_counts_df)

In [ ]:
from ephys_utils import AnalysisConfig, build_all_counts_df

cfg = AnalysisConfig(
    align_key="goCue",
    count_window_s=(0.0, 0.2),        # 0–200 ms post-cue response window
    baseline_window_s=(-1.0, 0.0),    # -1 to 0 s pre-cue baseline
    min_trials_per_group=20,
)

if ENV == "codeocean":
    all_counts_df = build_all_counts_df(units_with_spikes, cfg, base_dirs)
else:
    all_counts_df = pd.read_parquet(FOR_LOCAL / "all_counts_df.parquet")

print("all_counts_df shape :", all_counts_df.shape)
print("Columns             :", all_counts_df.columns.tolist())

## 4. Analysis specifications

Each `AnalysisSpec` captures the full provenance of one analysis:
predictor, response, method, trial filter, and transform.
Trial filtering is applied here — the OLS / Spearman functions receive
an already-filtered DataFrame and do no internal filtering themselves.

In [ ]:
from encoding_methods import AnalysisSpec, AnalysisResult, fit_encoding

# Default RT trial filter: exclude sub-50 ms (anticipatory) and >1 s (inattentive)
RT_QUERY = "reaction_time_firstmove > 0.05 and reaction_time_firstmove < 1.0"

specs = [
    AnalysisSpec(
        name="ols_rt_response",
        predictor_col="reaction_time_firstmove",
        response_col="spike_count",
        method="ols",
        trial_query=RT_QUERY,
        log_x=True, zscore_x=True,
        notes="Primary: log(RT) z-scored -> spike_count, 0-200 ms response window",
    ),
    AnalysisSpec(
        name="ols_rt_baseline",
        predictor_col="reaction_time_firstmove",
        response_col="baseline_spike_count",
        method="ols",
        trial_query=RT_QUERY,
        log_x=True, zscore_x=True,
        notes="Control: same predictor in baseline window (-1 to 0 s)",
    ),
    AnalysisSpec(
        name="spearman_rt_response",
        predictor_col="reaction_time_firstmove",
        response_col="spike_count",
        method="spearman",
        trial_query=RT_QUERY,
        notes="Spearman cross-check for OLS",
    ),
]


## 5. Run encoding analyses

In [ ]:
results = {}
for spec in specs:
    results[spec.name] = fit_encoding(all_counts_df, spec)

## 6. Register results

`PerUnitStatsRegistry` is the shared output format. Downstream notebooks
(spatial, behavioral comparison) load from here rather than re-running
the fits.

In [ ]:
from per_unit_stats_registry import PerUnitStatsRegistry
from aind_dynamic_foraging_behavior_video_analysis.ephys.tongue_ephys import get_session_prefix

reg = PerUnitStatsRegistry(get_session_prefix=get_session_prefix, alpha=0.05)

for spec in specs:
    reg.register(results[spec.name])

print(reg)


## 7. Population T-stat distributions

In [ ]:
fig, axes = plt.subplots(1, len(specs), figsize=(5 * len(specs), 4), sharey=False)

for ax, spec in zip(axes, specs):
    r = results[spec.name].stats
    t = r["T"].dropna().values
    sig = r.loc[r["T"].notna(), "sig_fdr"].values
    n_pos = int(((r["T"] > 0) & r["sig_fdr"]).sum())
    n_neg = int(((r["T"] < 0) & r["sig_fdr"]).sum())

    bins = np.histogram_bin_edges(t, bins=25)
    ax.hist(t[~sig], bins=bins, alpha=0.7, label="not sig")
    ax.hist(t[sig],  bins=bins, alpha=0.9, color="orange", label="sig (FDR)")
    ax.axvline(0, lw=1, ls="--", color="k")
    ax.set_title(f"{spec.name}\nn={len(t)}, +{n_pos} / -{n_neg} sig", fontsize=9)
    ax.set_xlabel("T-statistic")
    ax.legend(fontsize=7, frameon=False)

fig.suptitle("Per-unit T-stat distributions (FDR \u03b1=0.05)", fontsize=11)
plt.tight_layout()
plt.show()


In [ ]:
# OLS vs Spearman T comparison (response window)
r_ols  = results["ols_rt_response"].stats.set_index(["session_prefix", "unit"])["T"]
r_sp   = results["spearman_rt_response"].stats.set_index(["session_prefix", "unit"])["T"]
merged = r_ols.rename("T_ols").to_frame().join(r_sp.rename("T_spearman"), how="inner").dropna()

from scipy.stats import spearmanr as _spearmanr
rho, p = _spearmanr(merged["T_ols"], merged["T_spearman"])

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(merged["T_ols"], merged["T_spearman"], s=18, alpha=0.5, edgecolors="none")
lim = max(merged.abs().max()) * 1.05
ax.plot([-lim, lim], [-lim, lim], "k--", lw=0.8, alpha=0.5)
ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
ax.set_xlabel("OLS T (log RT z-scored)"); ax.set_ylabel("Spearman T (rho-derived)")
ax.set_title(f"OLS vs Spearman  rho={rho:.3f}  p={p:.2e}", fontsize=10)
ax.grid(True, ls=":", alpha=0.4)
plt.tight_layout()
plt.show()


## 8. RT window parameter sweep

Sweep rt_min × rt_max to test how sensitive encoding prevalence is to
the RT range included. Positive and negative encoding tracked separately
so that loss of units (trial count dropout) is distinguishable from
genuine loss of encoding.

In [ ]:
RT_MINS = [0.05, 0.08, 0.10, 0.12, 0.15, 0.20]
RT_MAXS = [0.30, 0.40, 0.50, 0.60, 0.80, 1.00]

sweep_rows = []
for rt_min, rt_max in itertools.product(RT_MINS, RT_MAXS):
    if rt_min >= rt_max:
        continue
    sweep_spec = AnalysisSpec(
        name=f"sweep",
        predictor_col="reaction_time_firstmove",
        response_col="spike_count",
        method="ols",
        trial_query=f"reaction_time_firstmove >= {rt_min} and reaction_time_firstmove <= {rt_max}",
        log_x=True, zscore_x=True,
    )
    with contextlib.redirect_stdout(io.StringIO()):
        res = fit_encoding(all_counts_df, sweep_spec)
    r = res.stats
    valid   = r["T"].notna()
    sig     = valid & (r["q"] < 0.05)
    pos_sig = sig & (r["coef"] > 0)
    neg_sig = sig & (r["coef"] < 0)
    n_valid = int(valid.sum())
    n_pos   = int(pos_sig.sum())
    n_neg   = int(neg_sig.sum())
    sweep_rows.append({
        "rt_min": rt_min, "rt_max": rt_max,
        "n_valid":       n_valid,
        "pct_pos_sig":   100 * n_pos / n_valid if n_valid else np.nan,
        "pct_neg_sig":   100 * n_neg / n_valid if n_valid else np.nan,
        "mean_T_pos_sig": float(r.loc[pos_sig, "T"].mean()) if n_pos else np.nan,
        "mean_T_neg_sig": float(r.loc[neg_sig, "T"].mean()) if n_neg else np.nan,
    })
    print(f"rt=[{rt_min:.2f}, {rt_max:.2f}]  valid={n_valid}  "
          f"+{n_pos}({100*n_pos/n_valid if n_valid else 0:.0f}%)  "
          f"-{n_neg}({100*n_neg/n_valid if n_valid else 0:.0f}%)")

sweep_df = pd.DataFrame(sweep_rows)


In [ ]:
def pivot_sweep(df, col):
    return df.pivot(index="rt_max", columns="rt_min", values=col)

panels = [
    ("n_valid",        "# units analyzed (≥ min trials)", "Greens", "{:.0f}"),
    ("pct_pos_sig",    "% pos-sig (more spikes → longer RT)", "Blues",  "{:.0f}"),
    ("pct_neg_sig",    "% neg-sig (fewer spikes → longer RT)", "Reds",   "{:.0f}"),
    (None,             "",                                     None,     ""),
    ("mean_T_pos_sig", "Mean T | pos-sig units",              "Blues",  "{:.2f}"),
    ("mean_T_neg_sig", "Mean T | neg-sig units",              "Reds_r", "{:.2f}"),
]

fig, axes = plt.subplots(2, 3, figsize=(17, 8))
for ax, (col, title, cmap, fmt) in zip(axes.flat, panels):
    if col is None:
        ax.set_visible(False)
        continue
    mat = pivot_sweep(sweep_df, col)
    vals = mat.values[np.isfinite(mat.values)]
    vmin, vmax = vals.min(), vals.max()
    im = ax.imshow(mat.values, cmap=cmap, aspect="auto", origin="upper", vmin=vmin, vmax=vmax)
    ax.set_xticks(range(len(mat.columns)))
    ax.set_xticklabels([f"{v:.2f}" for v in mat.columns], fontsize=8)
    ax.set_yticks(range(len(mat.index)))
    ax.set_yticklabels([f"{v:.2f}" for v in mat.index], fontsize=8)
    ax.set_xlabel("rt_min (s)", fontsize=9); ax.set_ylabel("rt_max (s)", fontsize=9)
    ax.set_title(title, fontsize=9)
    plt.colorbar(im, ax=ax, shrink=0.8)
    mid = (vmin + vmax) / 2
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            v = mat.values[i, j]
            if np.isfinite(v):
                light = abs(v - mid) < (vmax - vmin) * 0.3
                ax.text(j, i, fmt.format(v), ha="center", va="center",
                        fontsize=7, color="black" if light else "white")

fig.suptitle("RT window sweep — spike count ~ log(RT)", fontsize=12)
fig.tight_layout()
plt.show()

## 9. Population tuning curve

E[spike_count | log-RT bin], z-scored per unit, split by OLS significance group.
Shows whether pos-sig and neg-sig classifications reflect genuinely different
population-level tuning.

In [ ]:
RT_MIN_CURVE, RT_MAX_CURVE, N_BINS = 0.05, 1.0, 21

df_curve = (
    all_counts_df
    .dropna(subset=["reaction_time_firstmove", "spike_count"])
    .query("reaction_time_firstmove >= @RT_MIN_CURVE and reaction_time_firstmove <= @RT_MAX_CURVE and reaction_time_firstmove > 0")
    .copy()
)
df_curve["log_rt"] = np.log(df_curve["reaction_time_firstmove"])
df_curve["session_prefix"] = df_curve["session"].map(get_session_prefix)
df_curve["unit_canon"] = df_curve["unit_id"].apply(
    lambda x: str(int(float(x))) if pd.notna(x) else str(x)
)

# z-score per unit
u_stats = df_curve.groupby(["session_prefix", "unit_canon"])["spike_count"].agg(["mean", "std"])
df_curve = df_curve.join(
    u_stats.rename(columns={"mean": "_u_mean", "std": "_u_std"}),
    on=["session_prefix", "unit_canon"]
)
df_curve["sc_z"] = (df_curve["spike_count"] - df_curve["_u_mean"]) / df_curve["_u_std"].replace(0, np.nan)

# quantile bins in log-RT space
df_curve["rt_bin"] = pd.qcut(df_curve["log_rt"], N_BINS, labels=False)
bin_rt_center = df_curve.groupby("rt_bin")["reaction_time_firstmove"].median()

# per-unit mean per bin
unit_bin = (
    df_curve.groupby(["session_prefix", "unit_canon", "rt_bin"])["sc_z"]
    .mean()
    .unstack("rt_bin")
)
bin_cols = list(unit_bin.columns)

# attach group from OLS response fit
r_main = results["ols_rt_response"].stats.copy()
r_main["group"] = "neither"
r_main.loc[(r_main["q"] < 0.05) & (r_main["coef"] > 0), "group"] = "pos_sig"
r_main.loc[(r_main["q"] < 0.05) & (r_main["coef"] < 0), "group"] = "neg_sig"
group_map = r_main.set_index(["session_prefix", "unit"])["group"].to_dict()
unit_bin["group"] = [group_map.get((sp, u), "neither") for sp, u in unit_bin.index]

GROUPS_CURVE = {
    "all":     (unit_bin[bin_cols],                                     "black",   "All units"),
    "pos_sig": (unit_bin.loc[unit_bin["group"]=="pos_sig", bin_cols],   "#3a7abf", "Pos-sig"),
    "neg_sig": (unit_bin.loc[unit_bin["group"]=="neg_sig", bin_cols],   "#c0392b", "Neg-sig"),
    "neither": (unit_bin.loc[unit_bin["group"]=="neither", bin_cols],   "#e67e22", "Neither"),
}

fig, ax = plt.subplots(figsize=(8, 5))
x = bin_rt_center.values
for key, (mat, color, label) in GROUPS_CURVE.items():
    if len(mat) == 0:
        continue
    m = mat.mean(axis=0).values
    s = mat.sem(axis=0).values
    lw_fit = sm_lowess(m, x, frac=0.45, return_sorted=False)
    lw = 2.5 if key == "all" else 1.8
    ls = "--" if key == "all" else "-"
    ax.fill_between(x, m - s, m + s, alpha=0.12, color=color)
    ax.plot(x, m, "o", color=color, ms=3.5, alpha=0.5)
    ax.plot(x, lw_fit, ls, color=color, lw=lw, label=f"{label} (n={len(mat)})")

ax.axhline(0, color="grey", lw=0.8, ls=":")
ax.set_xscale("log")
xticks = [0.05, 0.1, 0.2, 0.3, 0.5, 0.7, 1.0]
ax.set_xticks(xticks); ax.set_xticklabels([f"{v*1000:.0f}" for v in xticks])
ax.set_xlabel("Reaction time (ms)", fontsize=12)
ax.set_ylabel("Spike count (z-score)", fontsize=12)
ax.set_title("Population tuning curve: spike count vs RT", fontsize=12)
ax.legend(fontsize=9, frameon=False)
plt.tight_layout()
plt.show()


## 10. Early vs late window classification

Classify units independently in an early [0.05, 0.30 s] and late [0.12, 1.00 s]
RT window using separate OLS fits. Groups: neg-early only, pos-late only, both, neither.

These windows were chosen to capture: (1) units that respond strongly to fast RTs
(neg-early) and (2) units that scale positively with slower RTs (pos-late).
Zero overlap between groups indicates two distinct monotonic populations.

In [ ]:
EARLY_QUERY = "reaction_time_firstmove >= 0.05 and reaction_time_firstmove <= 0.30"
LATE_QUERY  = "reaction_time_firstmove >= 0.12 and reaction_time_firstmove <= 1.00"

spec_early = AnalysisSpec("_early", "reaction_time_firstmove", "spike_count",
                          "ols", EARLY_QUERY, log_x=True, zscore_x=True)
spec_late  = AnalysisSpec("_late",  "reaction_time_firstmove", "spike_count",
                          "ols", LATE_QUERY,  log_x=True, zscore_x=True)

with contextlib.redirect_stdout(io.StringIO()):
    res_early = fit_encoding(all_counts_df, spec_early)
    res_late  = fit_encoding(all_counts_df, spec_late)

def sign_label(r, alpha=0.05):
    out = r[["session_prefix", "unit"]].copy()
    sig = r["q"] < alpha
    out["sign"] = "neither"
    out.loc[sig & (r["coef"] > 0), "sign"] = "pos"
    out.loc[sig & (r["coef"] < 0), "sign"] = "neg"
    return out.set_index(["session_prefix", "unit"])["sign"]

early_sign = sign_label(res_early.stats)
late_sign  = sign_label(res_late.stats)

neg_early_units = set(early_sign[early_sign == "neg"].index)
pos_late_units  = set(late_sign[late_sign == "pos"].index)
overlap_units   = neg_early_units & pos_late_units

print(f"Neg-sig early [0.05, 0.30 s] : {len(neg_early_units)}")
print(f"Pos-sig late  [0.12, 1.00 s] : {len(pos_late_units)}")
print(f"Overlap (both)               : {len(overlap_units)}")


In [ ]:
def get_group(sp_u):
    in_neg = sp_u in neg_early_units
    in_pos = sp_u in pos_late_units
    if in_neg and in_pos: return "both"
    if in_neg:            return "neg_early"
    if in_pos:            return "pos_late"
    return "neither"

unit_bin2 = unit_bin[bin_cols].copy()
unit_bin2["group"] = [get_group((sp, u)) for sp, u in unit_bin2.index]

GROUPS_EL = {
    "neg_early": ("#c0392b", "Neg-sig early [0.05, 0.30 s]"),
    "pos_late":  ("#3a7abf", "Pos-sig late  [0.12, 1.00 s]"),
    "both":      ("#8e44ad", "Both"),
    "neither":   ("#aaaaaa", "Neither"),
}

fig, ax = plt.subplots(figsize=(9, 5))
x = bin_rt_center.values

# All-units reference
m_all = unit_bin[bin_cols].mean(axis=0).values
ax.plot(x, sm_lowess(m_all, x, frac=0.45, return_sorted=False),
        "--", color="black", lw=2, label=f"All units (n={len(unit_bin)})", zorder=5)

for key, (color, label) in GROUPS_EL.items():
    mat = unit_bin2.loc[unit_bin2["group"] == key, bin_cols]
    if len(mat) == 0: continue
    m = mat.mean(axis=0).values
    s = mat.sem(axis=0).values
    ax.fill_between(x, m - s, m + s, alpha=0.12, color=color)
    ax.plot(x, m, "o", color=color, ms=3.5, alpha=0.5)
    ax.plot(x, sm_lowess(m, x, frac=0.45, return_sorted=False),
            "-", color=color, lw=2, label=f"{label} (n={len(mat)})")

# Union average (neg-early ∪ pos-late)
mat_union = unit_bin2.loc[unit_bin2["group"].isin(["neg_early","pos_late","both"]), bin_cols]
if len(mat_union):
    m_u = mat_union.mean(axis=0).values
    ax.plot(x, sm_lowess(m_u, x, frac=0.45, return_sorted=False),
            "-", color="#2ecc71", lw=2.5, zorder=6,
            label=f"Avg: neg-early ∪ pos-late (n={len(mat_union)})")

ax.axhline(0, color="grey", lw=0.8, ls=":")
ax.axvline(0.30, color="grey", lw=0.8, ls=":", alpha=0.6)
ax.axvline(0.12, color="grey", lw=0.8, ls=":", alpha=0.6)
ax.set_xscale("log")
ax.set_xticks(xticks); ax.set_xticklabels([f"{v*1000:.0f}" for v in xticks])
ax.set_xlabel("Reaction time (ms)", fontsize=12)
ax.set_ylabel("Spike count (z-score)", fontsize=12)
ax.set_title("Tuning curves: neg-sig early vs pos-sig late", fontsize=12)
ax.legend(fontsize=8, frameon=False, loc="upper left")
plt.tight_layout()
plt.show()

## 11. Response vs baseline

Compare T-stats between the response window (0–200 ms post-cue) and the
pre-cue baseline (-1 to 0 s). Units with response encoding but not baseline
encoding are of primary interest.

In [ ]:
r_resp = results["ols_rt_response"].stats.set_index(["session_prefix","unit"])[["T","sig_fdr"]].rename(columns={"T":"T_response","sig_fdr":"sig_response"})
r_base = results["ols_rt_baseline"].stats.set_index(["session_prefix","unit"])[["T","sig_fdr"]].rename(columns={"T":"T_baseline","sig_fdr":"sig_baseline"})
rb = r_resp.join(r_base, how="inner").dropna()

fig, ax = plt.subplots(figsize=(5, 5))
colors = np.where(rb["sig_response"] & ~rb["sig_baseline"], "#e74c3c",
         np.where(rb["sig_response"] & rb["sig_baseline"],  "#9b59b6",
         np.where(~rb["sig_response"] & rb["sig_baseline"], "#3498db", "#aaaaaa")))
ax.scatter(rb["T_response"], rb["T_baseline"], c=colors, s=18, alpha=0.6, edgecolors="none")
lim = max(rb[["T_response","T_baseline"]].abs().max()) * 1.05
ax.axhline(0, lw=0.8, ls="--", color="k"); ax.axvline(0, lw=0.8, ls="--", color="k")
ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
ax.set_xlabel("T \u2014 response window (0\u2013200 ms)")
ax.set_ylabel("T \u2014 baseline window (-1 to 0 s)")
ax.set_title("Response vs baseline encoding")

n_resp_only = int((rb["sig_response"] & ~rb["sig_baseline"]).sum())
n_base_only = int((~rb["sig_response"] & rb["sig_baseline"]).sum())
n_both      = int((rb["sig_response"] & rb["sig_baseline"]).sum())
from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(color="#e74c3c", label=f"Response only (n={n_resp_only})"),
    Patch(color="#9b59b6", label=f"Both (n={n_both})"),
    Patch(color="#3498db", label=f"Baseline only (n={n_base_only})"),
    Patch(color="#aaaaaa", label="Neither"),
], fontsize=8, frameon=False)
plt.tight_layout()
plt.show()


## 12. Poisson GLM encoding

Poisson GLM: `spike_count ~ 1 + log(RT)`, per unit. Compared against OLS T-stats.
The GLM accounts for the count-data nature of spike responses (non-negative integers,
variance proportional to mean). Coefficient is on the log scale: change in
log(mean spike count) per SD of log(RT).

In [ ]:
spec_glm = AnalysisSpec(
    name="glm_poisson_rt",
    predictor_col="reaction_time_firstmove",
    response_col="spike_count",
    method="glm",
    glm_family="poisson",
    trial_query=RT_QUERY,
    log_x=True, zscore_x=True,
    notes="Poisson GLM: log(mean spike count) ~ log(RT) z-scored",
)

res_glm = fit_encoding(all_counts_df, spec_glm)
reg.register(res_glm)
print(res_glm)

# Compare OLS vs GLM T-stats
t_ols = results["ols_rt_response"].stats.set_index(["session_prefix", "unit"])["T"]
t_glm = res_glm.stats.set_index(["session_prefix", "unit"])["T"]
cmp = t_ols.rename("T_ols").to_frame().join(t_glm.rename("T_glm"), how="inner").dropna()

from scipy.stats import spearmanr as _spearmanr
rho, p_rho = _spearmanr(cmp["T_ols"], cmp["T_glm"])

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Left: T-stat distribution for GLM
ax = axes[0]
r = res_glm.stats
t = r["T"].dropna().values
sig = r.loc[r["T"].notna(), "sig_fdr"].values
n_pos = int(((r["T"] > 0) & r["sig_fdr"]).sum())
n_neg = int(((r["T"] < 0) & r["sig_fdr"]).sum())
bins = np.histogram_bin_edges(t, bins=25)
ax.hist(t[~sig], bins=bins, alpha=0.7, label="not sig")
ax.hist(t[sig],  bins=bins, alpha=0.9, color="orange", label="sig (FDR)")
ax.axvline(0, lw=1, ls="--", color="k")
ax.set_title(f"Poisson GLM T-stats\nn={len(t)}, +{n_pos} / -{n_neg} sig", fontsize=10)
ax.set_xlabel("T-statistic (Wald)")
ax.legend(fontsize=8, frameon=False)

# Right: OLS vs GLM T-stat scatter
ax = axes[1]
ax.scatter(cmp["T_ols"], cmp["T_glm"], s=18, alpha=0.5, edgecolors="none")
lim = max(cmp.abs().max()) * 1.05
ax.plot([-lim, lim], [-lim, lim], "k--", lw=0.8, alpha=0.5)
ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
ax.set_xlabel("OLS T (log RT z-scored)")
ax.set_ylabel("Poisson GLM T (Wald)")
ax.set_title(f"OLS vs Poisson GLM  rho={rho:.3f}  p={p_rho:.2e}", fontsize=10)
ax.grid(True, ls=":", alpha=0.4)

plt.tight_layout()
plt.show()
